# Model Validation & Hyperparameter Tuning 🔬

Building a model is easy. Building one that actually works on new data? That's the real challenge. Learn to validate properly and tune hyperparameters.

## What You'll Learn
- Train/validation/test split strategy
- Cross-validation (k-fold, stratified, time-series)
- Hyperparameter tuning (GridSearchCV, RandomizedSearchCV)
- Learning curves and overfitting detection
- Model selection and comparison
- Real-world validation workflows

## The Train/Validation/Test Split 📊

### The Problem

```
❌ WRONG WAY:
Train on all data → Test on same data
Result: 99% accuracy (but fails in production!)

✅ RIGHT WAY:
Train on 70% → Validate on 15% → Test on 15%
Result: 78% on test (real-world performance)
```

### The Three Sets

1. **Training Set (70%):** What the model learns from
2. **Validation Set (15%):** What we tune hyperparameters with
3. **Test Set (15%):** Final evaluation (touch ONLY once at the end!)

### Golden Rule

**Never tune on test data!** Once you look at test data, it's contaminated. Save it for final evaluation only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.datasets import make_classification, load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

print("📚 Libraries loaded!")

In [ ]:
# Create dataset
X, y = make_classification(n_samples=300, n_features=10, n_informative=5,
                          n_redundant=2, random_state=42)

# Visualize the wrong way
print("="*60)
print("❌ WRONG: Evaluate on training data")
print("="*60)

model_wrong = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
model_wrong.fit(X, y)
train_score_wrong = model_wrong.score(X, y)

print(f"\nTraining Accuracy: {train_score_wrong:.1%}")
print(f"\n⚠️  Problem: We don't know real performance on NEW data!")
print(f"This model will likely FAIL in production.")

In [ ]:
# The right way
print("\n" + "="*60)
print("✅ RIGHT: Use train/validation/test split")
print("="*60)

# Step 1: Split off test set (and never touch it until the end!)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Step 2: Split remaining into train and validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)
# Note: 0.176 ≈ 15/85, so we get roughly 70% train, 15% val, 15% test

# Train model
model_right = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_right.fit(X_train, y_train)

# Evaluate on each set
train_acc = model_right.score(X_train, y_train)
val_acc = model_right.score(X_val, y_val)
test_acc = model_right.score(X_test, y_test)

print(f"\nTraining Accuracy: {train_acc:.1%}")
print(f"Validation Accuracy: {val_acc:.1%}")
print(f"Test Accuracy: {test_acc:.1%}")
print(f"\n✅ Test accuracy is most important (final evaluation)")
print(f"✅ Validation accuracy guides tuning")
print(f"✅ Gap between train and test = overfitting")

## Cross-Validation 🔄

### The Problem with Single Split

- Results depend on which data points end up in train vs test
- One bad train/test split can mislead you

### K-Fold Cross-Validation

```
Fold 1: Train on [2,3,4,5]  Validate on [1]
Fold 2: Train on [1,3,4,5]  Validate on [2]
Fold 3: Train on [1,2,4,5]  Validate on [3]
Fold 4: Train on [1,2,3,5]  Validate on [4]
Fold 5: Train on [1,2,3,4]  Validate on [5]

Final Score = Average of all 5 fold scores
```

### Why This Works

- Every data point used for validation exactly once
- More stable estimate of real performance
- Better use of limited data

In [ ]:
# K-Fold Cross-Validation
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

print("="*60)
print("K-FOLD CROSS-VALIDATION")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=42)

# Perform cross-validation
cv_scores = cross_val_score(model, X_temp, y_temp, cv=kfold, scoring='accuracy')

print(f"\nIndividual fold scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.1%}")

print(f"\nMean CV Score: {cv_scores.mean():.1%}")
print(f"Std Dev:       {cv_scores.std():.1%}")
print(f"Range:         {cv_scores.min():.1%} to {cv_scores.max():.1%}")

print(f"\n✅ More robust estimate than single split")
print(f"✅ Shows if performance is stable across different data")

In [ ]:
# Stratified K-Fold (for imbalanced data)
print("\n" + "="*60)
print("STRATIFIED K-FOLD (for imbalanced data)")
print("="*60)

# Create imbalanced dataset
from sklearn.datasets import make_classification
X_imb, y_imb = make_classification(n_samples=300, weights=[0.9, 0.1], 
                                   n_features=10, random_state=42)

print(f"\nClass distribution:")
print(f"  Class 0: {(y_imb == 0).sum()} ({(y_imb == 0).sum()/len(y_imb)*100:.0f}%)")
print(f"  Class 1: {(y_imb == 1).sum()} ({(y_imb == 1).sum()/len(y_imb)*100:.0f}%)")

# Regular k-fold
kfold_regular = KFold(n_splits=5, shuffle=True, random_state=42)
scores_regular = cross_val_score(model, X_imb, y_imb, cv=kfold_regular)

# Stratified k-fold
kfold_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_stratified = cross_val_score(model, X_imb, y_imb, cv=kfold_stratified)

print(f"\nRegular K-Fold:")
print(f"  Mean: {scores_regular.mean():.1%}")
print(f"  Std:  {scores_regular.std():.1%}")

print(f"\nStratified K-Fold:")
print(f"  Mean: {scores_stratified.mean():.1%}")
print(f"  Std:  {scores_stratified.std():.1%}")

print(f"\n✅ Stratified maintains class proportions in each fold")
print(f"✅ Better for imbalanced datasets")

## Hyperparameter Tuning 🎯

### Parameters vs Hyperparameters

- **Parameters:** Learned from data (weights in neural networks)
- **Hyperparameters:** Set before training (learning rate, tree depth)

### Common Hyperparameters

**Random Forest:**
- `n_estimators`: Number of trees (10, 50, 100, 200)
- `max_depth`: Tree depth (3, 5, 10, None)
- `min_samples_split`: Min samples to split (2, 5, 10)

**XGBoost:**
- `n_estimators`: Boosting rounds (100, 200, 500)
- `max_depth`: Tree depth (3, 5, 7)
- `learning_rate`: Step size (0.01, 0.05, 0.1)

### Tuning Methods

1. **GridSearchCV:** Try all combinations (exhaustive)
2. **RandomizedSearchCV:** Random sample of combinations (faster)
3. **Bayesian Optimization:** Smart exploration (most efficient)

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Define hyperparameter grid
param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10]
}

print("="*60)
print("GRIDSEARCHCV: EXHAUSTIVE SEARCH")
print("="*60)

# Calculate total combinations
n_combinations = len(param_grid['n_estimators']) * len(param_grid['max_depth']) * len(param_grid['min_samples_split'])
print(f"\nTotal combinations to try: {n_combinations}")
print(f"With 5-fold CV: {n_combinations * 5} model fits")
print(f"(This will take a minute or two...)\n")

# GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # Use all CPU cores
)

grid_search.fit(X_temp, y_temp)

print(f"✅ Search complete!")
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.1%}")

In [ ]:
# Compare top results
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score').head(10)

print(f"\nTop 10 Parameter Combinations:")
print("-" * 80)

for idx, row in results_df.iterrows():
    rank = row['rank_test_score']
    score = row['mean_test_score']
    params = row['params']
    print(f"Rank {int(rank):2d}: Score {score:.1%} | n_est={params['n_estimators']:3d} depth={str(params['max_depth']):4s} min_split={params['min_samples_split']}")

In [ ]:
# RandomizedSearchCV (faster for large grids)
print("\n" + "="*60)
print("RANDOMIZEDSEARCHCV: RANDOM SAMPLING")
print("="*60)

# Larger grid (would be too slow with GridSearchCV)
param_dist = {
    'n_estimators': [10, 20, 50, 100, 150, 200, 300],
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 3, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

total_combinations_random = 7 * 7 * 4 * 3 * 2
print(f"\nTotal possible combinations: {total_combinations_random}")
print(f"Testing only 20 random combinations with 5-fold CV")
print(f"(Much faster than GridSearchCV!)\n")

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=20,  # Test 20 random combinations
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_temp, y_temp)

print(f"✅ Search complete!")
print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.1%}")

## Learning Curves 📈

### What Are They?

Plots showing training and validation score vs amount of training data.

### What They Reveal

**Underfitting (High Bias):**
```
       Score
    100% ┌─────────────────
        │ \
        │  \ Train
     75% │   \
        │    ┣━━━━━━━ Val (stuck low)
     50% ┃    \
        │     \
      0% └──────────────── Data Size

Problem: Model too simple, can't learn patterns
Solution: Increase complexity
```

**Overfitting (High Variance):**
```
       Score
    100% ┌──── Train
        │ ╱
     75% ┃  ╱
        │  ╱  Val
     50% │ ╱─────── (stuck low)
        │╱
      0% └──────────────── Data Size

Problem: Model memorizes, doesn't generalize
Solution: Add data or simplify model
```

**Good Fit:**
```
       Score
    100% ┌────────── Train
        │ ╱
     75% ┃╱ ╔════════ Val
        │╱  ║
     50% ┃   ║ (gap stable)
        │
      0% └──────────────── Data Size

Small stable gap = good generalization
```

In [ ]:
from sklearn.model_selection import learning_curve

# Generate dataset
X_learn, y_learn = make_classification(n_samples=500, n_features=10, 
                                       n_informative=5, random_state=42)

# Test different complexities
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

complexities = [
    {'max_depth': 2, 'n_estimators': 10, 'title': 'Underfitting (Too Simple)'},
    {'max_depth': 10, 'n_estimators': 50, 'title': 'Good Fit'},
    {'max_depth': 30, 'n_estimators': 100, 'title': 'Overfitting (Too Complex)'}
]

for ax, config in zip(axes, complexities):
    model = RandomForestClassifier(
        max_depth=config['max_depth'],
        n_estimators=config['n_estimators'],
        random_state=42
    )
    
    # Get learning curve
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_learn, y_learn, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, 
        random_state=42,
        n_jobs=-1
    )
    
    train_mean = np.mean(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    # Plot
    ax.plot(train_sizes, train_mean, marker='o', label='Training Score', linewidth=2)
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
    
    ax.plot(train_sizes, val_mean, marker='s', label='Validation Score', linewidth=2)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
    
    ax.set_title(config['title'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.4, 1.0])

plt.tight_layout()
plt.show()

print("Notice the different patterns:")
print("  • Left: Gap stays large (needs more complexity)")
print("  • Middle: Gap is small and stable (good!)")
print("  • Right: Large gap that grows (overfitting!)")

## Validation Metrics Beyond Accuracy 📊

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve
)

# Train model and make predictions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("="*60)
print("VALIDATION METRICS")
print("="*60)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n📊 Classification Metrics:")
print(f"  Accuracy:  {accuracy:.1%}  (% correct predictions)")
print(f"  Precision: {precision:.1%}  (% predicted positive that are correct)")
print(f"  Recall:    {recall:.1%}  (% actual positive that we caught)")
print(f"  F1-Score:  {f1:.1%}  (harmonic mean of precision & recall)")
print(f"\n🎯 ROC-AUC:    {roc_auc:.1%}  (ranking quality, 1.0 = perfect)")

print(f"\n💡 Use different metrics based on problem:")
print(f"   - Balanced classes → Accuracy")
print(f"   - Imbalanced classes → F1, Precision, Recall")
print(f"   - Ranking quality → ROC-AUC")
print(f"   - Cost matters → Choose based on what's expensive")

In [ ]:
# ROC Curve visualization
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC Curve
axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC={roc_auc:.1%})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall_curve, precision_curve, linewidth=2, label='Precision-Recall')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## Complete Validation Workflow 🔄

In [ ]:
print("="*70)
print("COMPLETE MODEL VALIDATION WORKFLOW")
print("="*70)

workflow = """
┌─────────────────────────────────────────────────────────────┐
│  1. SPLIT DATA (70% train, 15% val, 15% test)              │
│     Never touch test until the very end!                   │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  2. SETUP HYPERPARAMETER GRID                              │
│     Define ranges for each hyperparameter                  │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  3. CROSS-VALIDATION (5 or 10 fold)                        │
│     Search hyperparameter space                            │
│     GridSearchCV or RandomizedSearchCV                     │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  4. SELECT BEST HYPERPARAMETERS                            │
│     Choose based on CV score                               │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  5. RETRAIN ON FULL TRAIN+VAL SET                          │
│     With best hyperparameters                              │
│     Now we can use validation set for training             │
└─────────────────────────────────────────────────────────────┘
                           ↓
┌─────────────────────────────────────────────────────────────┐
│  6. EVALUATE ON TEST SET (ONCE)                            │
│     This is your FINAL PERFORMANCE ESTIMATE                │
│     Report this number (not CV score!)                     │
└─────────────────────────────────────────────────────────────┘

⚠️  KEY POINTS:
  • Don't tune on test data (ever!)
  • Use cross-validation for stable estimates
  • Report test performance, not training performance
  • If test << CV score, you overfit (try simpler model)
  • If test ≈ CV score, you generalize well!
"""

print(workflow)

In [ ]:
# Let's execute this workflow

# Step 1: Split data
print("Step 1: Split Data")
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)
print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Step 2: Setup hyperparameter grid
print("\nStep 2: Setup Hyperparameter Grid")
param_grid = {
    'max_depth': [5, 10, 15],
    'n_estimators': [50, 100, 150]
}
print(f"  Testing {3*3} combinations with 5-fold CV")

# Step 3: Cross-validation
print("\nStep 3: Running GridSearchCV...")
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_temp, y_temp)  # Train+Val combined

# Step 4: Best parameters
print("\nStep 4: Best Hyperparameters")
print(f"  {grid_search.best_params_}")
print(f"  CV Score: {grid_search.best_score_:.1%}")

# Step 5: Retrain on full train+val
print("\nStep 5: Retraining on Full Train+Val Set")
best_model = RandomForestClassifier(
    **grid_search.best_params_,
    random_state=42
)
best_model.fit(X_temp, y_temp)  # Using all train+val
print("  Done!")

# Step 6: Evaluate on test (FINAL!)
print("\nStep 6: Final Evaluation on Test Set")
test_accuracy = best_model.score(X_test, y_test)
print(f"  ✅ TEST ACCURACY: {test_accuracy:.1%}")

print("\n" + "="*70)
print(f"Final Report: Model achieves {test_accuracy:.1%} accuracy on unseen test data")
print("="*70)

## Common Validation Mistakes ❌

In [ ]:
mistakes = """
❌ MISTAKE 1: Scaling before train/test split
   Wrong:
     scaler = StandardScaler()
     X_scaled = scaler.fit_transform(X)  # Fit on ALL data!
     X_train, X_test, ... = train_test_split(X_scaled, ...)
   
   Right:
     X_train, X_test, ... = train_test_split(X, ...)
     scaler = StandardScaler()
     X_train = scaler.fit_transform(X_train)  # Fit on train only
     X_test = scaler.transform(X_test)       # Apply to test

❌ MISTAKE 2: Selecting features based on test performance
   Wrong:
     Check correlation with test target
     Select features that seem important
   
   Right:
     Select features using ONLY training data
     Apply same selection to test

❌ MISTAKE 3: Reporting CV score as final performance
   Wrong: "My model got 92% (5-fold CV)"
   Right: "My model got 89% on held-out test set"
   
   (Test score is more realistic)

❌ MISTAKE 4: Using test set for any tuning decisions
   Wrong:
     Check test accuracy
     If not good enough, change hyperparameters
     Test again
   
   Right:
     Use CV on train+val for tuning
     Test ONCE at the end for final estimate

❌ MISTAKE 5: Not validating preprocessing
   Wrong:
     Use SimpleImputer on all data
     Fill NaN with mean of entire dataset
   
   Right:
     Fit imputer on training data only
     Apply to test data

✅ GOLDEN RULE:
   "Your test set is sacred. Touch it only once at the very end."
"""

print(mistakes)

## Model Comparison 🏆

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Train different models on same data
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

print("="*60)
print("MODEL COMPARISON")
print("="*60)

results = {}

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_temp, y_temp, cv=5, scoring='accuracy')
    
    # Train on all train+val
    model.fit(X_temp, y_temp)
    test_score = model.score(X_test, y_test)
    
    results[name] = {
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test': test_score
    }

# Display results
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('Test', ascending=False)

print("\n", results_df.to_string())

print(f"\n🏆 Winner: {results_df.index[0]} with {results_df['Test'].iloc[0]:.1%} test accuracy")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df))
width = 0.25

ax.bar(x - width, results_df['CV Mean'], width, label='CV Mean', alpha=0.8)
ax.bar(x, results_df['Test'], width, label='Test', alpha=0.8)
ax.bar(x + width, results_df['CV Std'], width, label='CV Std', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison: CV vs Test Performance')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print("\n💡 Notice:")
print("  • Test score is more realistic than CV score")
print("  • Smaller CV Std = more stable model")
print("  • Some models have high CV but lower test (overfit)")

## Key Takeaways 🎯

✅ Always split data into train/validation/test BEFORE any preprocessing

✅ Use cross-validation for stable hyperparameter tuning

✅ GridSearchCV for small parameter spaces, RandomizedSearchCV for large

✅ Learning curves reveal underfitting vs overfitting

✅ Choose appropriate metrics (accuracy, precision, recall, F1, ROC-AUC)

✅ Never tune on test data - test set is sacred!

✅ Report test performance, not training or CV performance

✅ Compare multiple models before choosing winner

## Practice Exercise 💪

1. **Load dataset:** Use iris or another dataset
2. **Split properly:** 70% train, 15% val, 15% test
3. **Hyperparameter tune:** Use GridSearchCV with 5-fold CV
4. **Plot learning curves:** Show bias vs variance
5. **Compare models:** Test 3+ different algorithms
6. **Report results:** Final test accuracy only

Challenge: Beat a baseline model by tuning hyperparameters effectively!

## Next Steps 🚀

→ **Unsupervised Learning:** Clustering, dimensionality reduction

→ **Imbalanced Data:** Handle skewed class distributions

→ **Production:** Deploy models, monitoring, retraining

→ **Real Projects:** Apply everything to Kaggle or business problems!